In [2]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.2)

In [4]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [5]:
def input_prompt(state:BlogState) ->BlogState:
    # fetch the title

    title = state["title"]

    # call the llm and generate the outcome 
    prompt = f'Generate the blog on the give topic - {title}'
    output = model.invoke(prompt).content

    # update the state 
    state["outline"] = output

    return state

In [6]:
def create_blog(state:BlogState)->BlogState:
    title = state['title']
    outline = state['outline']



    prompt = f'write a detaile blog on the title - {title} using the outline - {outline}'
    content = model.invoke(prompt).content

    state['content'] = content


    return state

In [7]:
graph = StateGraph(BlogState)

# adding the node 

graph.add_node("input_prompt",input_prompt)
graph.add_node("create_blog",create_blog)

# adding the edeg

graph.add_edge(START,'input_prompt')
graph.add_edge('input_prompt','create_blog')
graph.add_edge('create_blog',END)

# compileing the graph

workflow =graph.compile()

In [ ]:
final_outcome = BlogState(title="cricket")

output = workflow.invoke(final_outcome)

print(output["content"][0])
